# COVID-19 Retrospective: Spread, Waves, Severity and India's Experience

**Data:** Johns Hopkins CSSE COVID-19 time series (CC BY 4.0, final update 10 Mar 2023) | **Stack:** pandas, numpy, matplotlib/seaborn, plotly, scipy

**Business context.** A public-health research team wants a retrospective of the pandemic: how it spread, when waves peaked, which countries were hit hardest, how deadly it was in different places, and what India's experience looked like.

**Key questions**
1. How many cases and deaths were recorded globally, and when did the waves peak?
2. Which countries had the most cases, deaths and the highest fatality rates?
3. How did India's waves compare (Delta vs Omicron)?
4. How long after cases do deaths follow?
5. Which US states had the highest death rates per 100k?

**Run from the repo root** (needs `data/` with the five JHU CSVs). All charts are saved to `outputs/`.

In [1]:
import os, re, warnings, urllib.request, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (11, 5)})

DATA = os.environ.get("COVID_DATA", "data")
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

def save_mpl(name):
    plt.tight_layout()
    plt.savefig(f"{OUT}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def save_px(fig, name):
    fig.write_html(f"{OUT}/{name}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(f"{OUT}/{name}.png", scale=2)   # needs kaleido; HTML is always saved
    except Exception:
        pass
    fig.show()

## 1. Data loading and cleaning
Steps: merge provinces into countries, transpose wide to dates-as-rows, parse `m/d/yy` dates, convert cumulative to daily (clipping data corrections), drop non-country rows, flag impossible rows (deaths > confirmed), and merge population for per-million rates.

In [3]:
NON_COUNTRY = ["Diamond Princess", "MS Zaandam", "Summer Olympics 2020", "Winter Olympics 2022", "Antarctica"]

def load(kind):
    df = pd.read_csv(f"{DATA}/time_series_covid19_{kind}_global.csv")
    df = (df.drop(columns=["Province/State", "Lat", "Long"])
            .groupby("Country/Region").sum(numeric_only=True))          # 1. merge provinces into countries
    out = df.T                                                          # 2. wide -> dates as rows
    out.index = pd.to_datetime(out.index, format="%m/%d/%y")            # 3. parse "1/22/20"
    out.index.name = "date"
    return out

raw_conf_path = f"{DATA}/time_series_covid19_confirmed_global.csv"
if not os.path.exists(raw_conf_path):
    os.makedirs(DATA, exist_ok=True)
    url = (
        "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
        "csse_covid-19_data/csse_covid_19_time_series/"
        "time_series_covid19_confirmed_global.csv"
    )
    urls = [
        url,
        url.replace("/master/", "/main/"),
    ]
    for candidate in urls:
        try:
            urllib.request.urlretrieve(candidate, raw_conf_path)
            break
        except urllib.error.HTTPError as e:
            if e.code != 404:
                raise
    else:
        raise FileNotFoundError(
            "The JHU confirmed-cases CSV returned 404 on both master and main. "
            "Download the required JHU CSV files into the data/ directory, "
            "or update the source URL to a working mirror."
        )

raw_conf = pd.read_csv(raw_conf_path)
print(f"Province/State blank: {raw_conf['Province/State'].isna().mean():.0%} (single-row countries)")

conf_all, deaths_all = load("confirmed"), load("deaths")     # all rows (incl. cruise ships etc.) for raw-total checks
rec_all = load("recovered")
rec_total = rec_all.sum(axis=1)
print("Recovered series last reported on:", rec_total[rec_total > 0].index.max().date(),
      "| final-day total:", int(rec_total.iloc[-1]), "-> unusable after mid-2021, no 'active cases'.")

# Global (all rows, matches raw file totals)
new_cases_all  = conf_all.diff().fillna(0).clip(lower=0)
new_deaths_all = deaths_all.diff().fillna(0).clip(lower=0)

# Country-level frames (non-country rows removed)
conf   = conf_all.drop(columns=[c for c in NON_COUNTRY if c in conf_all.columns])
deaths = deaths_all.drop(columns=[c for c in NON_COUNTRY if c in deaths_all.columns])
new_cases  = conf.diff().fillna(0).clip(lower=0)                        # 4. cumulative -> daily, clip corrections
new_deaths = deaths.diff().fillna(0).clip(lower=0)

n_neg_c = int((conf_all.diff() < 0).sum().sum()); n_neg_d = int((deaths_all.diff() < 0).sum().sum())
print(f"Negative daily corrections clipped: {n_neg_c:,} in cases ({n_neg_c/conf_all.size:.3%} of cells), {n_neg_d:,} in deaths")
print("Countries after dropping non-country rows:", conf.shape[1])

# Deaths > confirmed: reporting errors -> flag and exclude from CFR
bad = deaths > conf
bad_countries = bad.any()
print(f"Deaths > confirmed on some dates: {bad_countries[bad_countries].index.tolist()} ({int(bad.sum().sum()):,} country-days excluded from CFR)")
cfr = (deaths / conf.where(conf > 0)).where(~bad)                       # cumulative CFR, clean

# Population for per-million rates
lk_path = f"{DATA}/UID_ISO_FIPS_LookUp_Table.csv"
LK_URL = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"
lk = pd.read_csv(lk_path if os.path.exists(lk_path) else LK_URL)
country_rows = lk[lk["Province_State"].isna()]
pop = country_rows.groupby("Country_Region")["Population"].sum()
pop_missing = lk[~lk["Country_Region"].isin(pop.index)].groupby("Country_Region")["Population"].sum()
pop = pd.concat([pop, pop_missing]); pop = pop[pop > 0]
iso3 = (lk.dropna(subset=["iso3"]).sort_values("Province_State", na_position="first")
          .drop_duplicates("Country_Region").set_index("Country_Region")["iso3"])
common = conf.columns.intersection(pop.index)
print(f"Countries with population: {len(common)} of {conf.shape[1]}; missing: {sorted(set(conf.columns)-set(common))}")
conf_pm, deaths_pm = conf[common] / pop[common] * 1e6, deaths[common] / pop[common] * 1e6

# 5. Tidy long table for plotting / merging
tidy = (conf.stack().rename("confirmed").to_frame()
          .join(deaths.stack().rename("deaths"))
          .join(new_cases.stack().rename("new_cases"))
          .join(new_deaths.stack().rename("new_deaths"))
          .reset_index().rename(columns={"Country/Region": "country"}))
tidy["cfr"] = np.where((tidy.confirmed > 0) & (tidy.deaths <= tidy.confirmed), tidy.deaths / tidy.confirmed.replace(0, np.nan), np.nan)
tidy.head()

FileNotFoundError: The JHU confirmed-cases CSV returned 404 on both master and main. Download the required JHU CSV files into the data/ directory, or update the source URL to a working mirror.

## 2. Feature engineering

In [ ]:
cases7  = new_cases.rolling(7, min_periods=1).mean()        # 10. 7-day averages remove weekend artefacts
deaths7 = new_deaths.rolling(7, min_periods=1).mean()
growth7 = conf.pct_change(7).replace([np.inf, -np.inf], np.nan)                       # weekly growth rate
doubling = (7 * np.log(2) / np.log(conf / conf.shift(7))).replace([np.inf, -np.inf], np.nan)   # doubling time (days)

def days_since_100(country):
    s = conf[country][conf[country] >= 100].reset_index(drop=True)
    s.index.name = "days_since_100th_case"
    return s

g_cases7  = new_cases_all.sum(axis=1).rolling(7, min_periods=1).mean()
g_deaths7 = new_deaths_all.sum(axis=1).rolling(7, min_periods=1).mean()
print("Engineered: new cases/deaths, 7d averages, CFR, per-million, weekly growth, doubling time, days since 100th case")

In [ ]:
assert conf_all.index.is_monotonic_increasing and conf_all.index.is_unique
assert (conf_all.diff().dropna() < 0).sum().sum() < 0.01 * conf_all.size, "corrections should be rare"
assert conf_all.iloc[-1].sum() == 676_570_149, f"last-day total is {conf_all.iloc[-1].sum():,}"
assert new_cases_all.sum().sum() + conf_all.iloc[0].sum() >= conf_all.iloc[-1].sum(), "clipping only adds"
assert (new_cases >= 0).all().all() and (new_deaths >= 0).all().all()
assert conf_all.shape == deaths_all.shape and (conf_all.index == deaths_all.index).all()
assert conf_all.shape[0] == 1143, f"expected 1,143 daily columns, got {conf_all.shape[0]}"
bad_list = bad_countries[bad_countries].index.tolist()
print("Deaths > cases (reporting errors, investigate):", bad_list)
print("All validation tests passed")

## 4. Exploratory data analysis
### 4.1 Global picture

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
tot_c, tot_d = conf_all.sum(axis=1), deaths_all.sum(axis=1)
ax[0, 0].plot(tot_c / 1e6, color="tab:blue");  ax[0, 0].set(title="Cumulative confirmed cases (millions)")
ax[0, 1].plot(tot_d / 1e6, color="tab:red");   ax[0, 1].set(title="Cumulative deaths (millions)")
for a, s, c, t, u in [(ax[1, 0], g_cases7, "tab:blue", "Daily new cases (7-day avg, millions)", 1e6),
                      (ax[1, 1], g_deaths7, "tab:red", "Daily new deaths (7-day avg, thousands)", 1e3)]:
    a.plot(s / u, color=c); a.set_title(t)
    pk = s.idxmax()
    a.annotate(f"Peak {pk:%d %b %Y}\n{s.max()/u:,.1f}", xy=(pk, s.max() / u), xytext=(-110, -30),
               textcoords="offset points", arrowprops=dict(arrowstyle="->"), fontsize=9)
plt.suptitle("Global COVID-19 trajectory", fontsize=14, y=1.01)
save_mpl("01_global_trajectory")

# Other local wave peaks (prominence-based)
from scipy.signal import find_peaks
pk_idx, _ = find_peaks(g_cases7.values, prominence=g_cases7.max() * 0.15, distance=60)
print("Case wave peaks (7d avg):")
print(g_cases7.iloc[pk_idx].round(0).to_string())

In [ ]:
yearly = pd.DataFrame({
    "new_cases":  new_cases_all.sum(axis=1).groupby(new_cases_all.index.year).sum(),
    "new_deaths": new_deaths_all.sum(axis=1).groupby(new_deaths_all.index.year).sum(),
    "cum_cases_year_end":  tot_c.groupby(tot_c.index.year).last(),
    "cum_deaths_year_end": tot_d.groupby(tot_d.index.year).last()})
yearly["CFR_%"] = yearly.new_deaths / yearly.new_cases * 100
yearly.index = [f"{y}" + (" (to 9 Mar)" if y == 2023 else "") for y in yearly.index]
display(yearly.style.format({"new_cases": "{:,.0f}", "new_deaths": "{:,.0f}", "cum_cases_year_end": "{:,.0f}",
                             "cum_deaths_year_end": "{:,.0f}", "CFR_%": "{:.2f}"}))
ax = yearly[["new_cases", "new_deaths"]].div(1e6).plot.bar(subplots=True, figsize=(9, 6), rot=0, legend=False,
                                                           color=["tab:blue", "tab:red"], title=["New cases (M)", "New deaths (M)"])
save_mpl("02_yearly_totals")

### 4.2 Countries

In [ ]:
last_c, last_d = conf.iloc[-1], deaths.iloc[-1]
dpm = deaths_pm.iloc[-1]
cfr_last = (last_d / last_c)[(last_c > 1e6) & (~bad.iloc[-1])]            # countries with > 1M cases, valid rows only

panels = [("Top 10 by confirmed cases (M)", last_c.nlargest(10) / 1e6, "tab:blue"),
          ("Top 10 by deaths (thousands)", last_d.nlargest(10) / 1e3, "tab:red"),
          ("Top 10 by deaths per million", dpm.nlargest(10), "tab:purple"),
          ("Top 10 CFR % (>1M cases)", cfr_last.nlargest(10) * 100, "tab:orange")]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for a, (t, s, c) in zip(axes.ravel(), panels):
    s.sort_values().plot.barh(ax=a, color=c); a.set_title(t); a.set_ylabel("")
save_mpl("03_top10_countries")

print("Lowest CFR (>1M cases):"); print((cfr_last.nsmallest(5) * 100).round(2).to_string())

In [ ]:
focus = [c for c in ["US", "India", "Brazil", "United Kingdom", "Italy", "Germany", "Korea, South", "South Africa"] if c in conf.columns]
fig, ax = plt.subplots(figsize=(11, 6))
for c in focus:
    days_since_100(c).plot(ax=ax, label=c)
ax.set_yscale("log"); ax.set(title="Cumulative cases since the 100th case (log scale)", xlabel="Days since 100th case", ylabel="Confirmed cases")
x = np.arange(0, 400)
for d in (3, 7):
    ax.plot(x, 100 * 2 ** (x / d), "k:", alpha=.35)
    ax.text(60 if d == 3 else 200, 100 * 2 ** ((60 if d == 3 else 200) / d), f"doubles every {d} d", fontsize=8, alpha=.6)
ax.set_ylim(100, None); ax.legend()
save_mpl("04_days_since_100th_case")

In [ ]:
last_pm = conf_pm.iloc[-1].rename("cases_per_million").to_frame().join(iso3).reset_index().dropna(subset=["iso3"])
fig = px.choropleth(last_pm, locations="iso3", color="cases_per_million", hover_name="Country/Region",
                    color_continuous_scale="YlOrRd", title="Confirmed cases per million (9 Mar 2023)")
save_px(fig, "05_choropleth_cases_per_million")

# Animated version (every 30 days + final day)
sel = list(range(0, len(conf_pm), 30)) + [len(conf_pm) - 1]
frames = conf_pm.iloc[sorted(set(sel))]
anim = frames.stack().rename("cases_per_million").reset_index()
anim["iso3"] = anim["Country/Region"].map(iso3); anim = anim.dropna(subset=["iso3"])
anim["date_str"] = anim["date"].dt.strftime("%Y-%m-%d")
fig = px.choropleth(anim, locations="iso3", color="cases_per_million", hover_name="Country/Region", animation_frame="date_str",
                    range_color=(0, float(conf_pm.iloc[-1].quantile(.95))), color_continuous_scale="YlOrRd",
                    title="Cumulative cases per million over time")
fig.write_html(f"{OUT}/06_choropleth_animated.html", include_plotlyjs="cdn")
fig.show()

### 4.3 India deep-dive: Delta vs Omicron

In [ ]:
IN = "India"
fig, ax = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
ax[0].plot(cases7[IN] / 1e3, color="tab:blue");  ax[0].set_title("India: daily new cases (7-day avg, thousands)")
ax[1].plot(deaths7[IN] / 1e3, color="tab:red");  ax[1].set_title("India: daily new deaths (7-day avg, thousands)")
roll_cfr = new_deaths[IN].rolling(28).sum() / new_cases[IN].rolling(28).sum() * 100
ax[2].plot(roll_cfr, label="rolling 28-day CFR %"); ax[2].plot(cfr[IN] * 100, label="cumulative CFR %")
ax[2].set_title("India: case fatality rate over time (%)"); ax[2].set_ylim(0, 6); ax[2].legend()
windows = {"Delta (2021)": ("2021-03-01", "2021-08-31"), "Omicron (2022)": ("2021-12-15", "2022-03-31")}
for a in ax[:2]:
    for (n, (s, e)), col in zip(windows.items(), ["orange", "green"]):
        a.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=.12, label=n)
ax[0].legend()
save_mpl("07_india_deepdive")

rows = []
for n, (s, e) in windows.items():
    c7, d7 = cases7[IN][s:e], deaths7[IN][s:e]
    wc, wd = new_cases[IN][s:e].sum(), new_deaths[IN][s:e].sum()
    rows.append({"wave": n, "peak_cases_7d": c7.max(), "peak_cases_date": c7.idxmax().date(),
                 "peak_deaths_7d": d7.max(), "peak_deaths_date": d7.idxmax().date(),
                 "window_cases": wc, "window_deaths": wd, "window_CFR_%": wd / wc * 100})
india_waves = pd.DataFrame(rows).set_index("wave")
display(india_waves.style.format({"peak_cases_7d": "{:,.0f}", "peak_deaths_7d": "{:,.0f}", "window_cases": "{:,.0f}",
                                  "window_deaths": "{:,.0f}", "window_CFR_%": "{:.2f}"}))
print(f"India final: {conf[IN].iloc[-1]:,.0f} cases, {deaths[IN].iloc[-1]:,.0f} deaths")

### 4.4 United States: states and counties

In [ ]:
us_conf   = pd.read_csv(f"{DATA}/time_series_covid19_confirmed_US.csv")
us_deaths = pd.read_csv(f"{DATA}/time_series_covid19_deaths_US.csv")
date_cols = [c for c in us_deaths.columns if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}", c)]
last_date = date_cols[-1]

us = (us_deaths[["UID", "FIPS", "Admin2", "Province_State", "Population", last_date]].rename(columns={last_date: "deaths"})
      .merge(us_conf[["UID", last_date]].rename(columns={last_date: "confirmed"}), on="UID", validate="1:1"))   # merge on UID
n_zero = int((us.Population == 0).sum())
print(f"Rows with Population = 0 excluded from per-capita rates: {n_zero}")
us = us[us.Population > 0].copy()
us["deaths_per_100k"] = us.deaths / us.Population * 1e5

TERRITORIES = ["American Samoa", "Guam", "Northern Mariana Islands", "Puerto Rico", "Virgin Islands", "Diamond Princess", "Grand Princess"]
states = us[~us.Province_State.isin(TERRITORIES)].groupby("Province_State")[["deaths", "confirmed", "Population"]].sum()
states["deaths_per_100k"] = states.deaths / states.Population * 1e5
states = states.sort_values("deaths_per_100k", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
states.deaths_per_100k.head(10).sort_values().plot.barh(ax=ax[0], color="tab:red");  ax[0].set_title("Highest COVID deaths per 100k (states)")
states.deaths_per_100k.tail(10).sort_values().plot.barh(ax=ax[1], color="tab:green"); ax[1].set_title("Lowest COVID deaths per 100k (states)")
for a in ax: a.set_ylabel("")
save_mpl("08_us_states_deaths_per_100k")
print(f"State range: {states.deaths_per_100k.min():.0f} to {states.deaths_per_100k.max():.0f} per 100k")
display(states.head(5).round(1))

In [ ]:
cty = us.dropna(subset=["FIPS"]).copy()
cty["fips5"] = cty.FIPS.astype(int).astype(str).str.zfill(5)
try:
    gj_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
    with urllib.request.urlopen(gj_url, timeout=30) as r:
        counties_geo = json.load(r)
    fig = px.choropleth(cty, geojson=counties_geo, locations="fips5", color="deaths_per_100k", scope="usa",
                        range_color=(0, float(cty.deaths_per_100k.quantile(.98))), color_continuous_scale="Reds",
                        hover_name="Admin2", hover_data=["Province_State", "deaths"],
                        title="COVID-19 deaths per 100k by county (9 Mar 2023)")
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    save_px(fig, "09_us_county_choropleth")
except Exception as e:
    print("County choropleth skipped (needs internet for the county GeoJSON):", e)

### 4.5 Relationships: case-to-death lag and seasonality

In [ ]:
wc, wd = new_cases_all.sum(axis=1), new_deaths_all.sum(axis=1)
def lagged(x, y, lags=range(0, 36)):
    out = []
    for L in lags:
        a, b = x.values[:len(x) - L], y.values[L:]
        r, p = stats.pearsonr(a, b)
        out.append((L, r, p))
    return pd.DataFrame(out, columns=["lag", "r", "p"]).set_index("lag")

lag_raw = lagged(wc, wd)
lag_sm  = lagged(wc.rolling(7, min_periods=1).mean(), wd.rolling(7, min_periods=1).mean())
best_raw, best_sm = lag_raw.r.idxmax(), lag_sm.r.idxmax()
print(f"Global daily series:   best lag = {best_raw} d, r = {lag_raw.r.max():.3f}")
print(f"Global 7-day averages: best lag = {best_sm} d, r = {lag_sm.r.max():.3f}")

# Per-country best lag (top 20 by cases, 7d-smoothed)
best = {}
for c in conf.iloc[-1].nlargest(20).index:
    t = lagged(cases7[c], deaths7[c]); best[c] = t.r.idxmax()
best = pd.Series(best); print(f"Per-country best lag (top-20 by cases): median {best.median():.0f} d, IQR {best.quantile(.25):.0f}-{best.quantile(.75):.0f} d")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(lag_raw.r, marker="o", ms=3, label="daily"); ax[0].plot(lag_sm.r, marker="o", ms=3, label="7-day avg")
ax[0].axvline(best_sm, ls="--", c="k", alpha=.4); ax[0].set(title="Cross-correlation: cases(t) vs deaths(t+lag), world", xlabel="Lag (days)", ylabel="Pearson r"); ax[0].legend()
best.sort_values().plot.barh(ax=ax[1], color="slategray"); ax[1].set(title="Best lag by country (days)", xlabel="days")
save_mpl("10_case_death_lag")

In [ ]:
m = wc.groupby([wc.index.year, wc.index.month]).sum().unstack()
m.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:m.shape[1]]
fig, ax = plt.subplots(figsize=(12, 3.8))
sns.heatmap(np.log10(m.replace(0, np.nan)), annot=(m / 1e6).round(1), fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "log10 new cases"}, ax=ax, linewidths=.5)
ax.set(title="Global new cases by month (millions; colour = log10 scale)", xlabel="Month", ylabel="Year")
save_mpl("11_monthly_seasonality_heatmap")

mm = (new_cases_all.sum(axis=1).groupby(new_cases_all.index.month).sum() / 1e6).round(1)
print("Cases by calendar month, all years pooled (M):", mm.to_dict())
print("Note: only 3 full years with distinct variants -> month-of-year effects are confounded by waves, not clean seasonality.")

## 5. Statistical tests
| Hypothesis | Method |
|---|---|
| H0: daily deaths don't follow daily cases | Lagged Pearson correlation, 0-35 days (section 4.5) |
| H0: CFR is the same in 2020 and 2022 | Two-proportion z-test |
| H0: per-capita death rates are equal across US states | Kruskal-Wallis on county rates grouped by state |
| Trend significance around each peak | Mann-Kendall (Kendall tau vs time) + regression slope |

In [ ]:
# A. Lagged correlation significance
print("H0: deaths don't follow cases ->",
      f"best lag {best_sm} d, r = {lag_sm.r.max():.3f}, p = {lag_sm.p[best_sm]:.2e}",
      "=> reject H0" if lag_sm.p[best_sm] < 0.05 else "=> fail to reject H0")
print("Caveat: autocorrelated time series inflate significance; read r as effect size.\n")

# B. Two-proportion z-test: CFR 2020 vs 2022
def two_prop_z(x1, n1, x2, n2):
    p1, p2, p = x1 / n1, x2 / n2, (x1 + x2) / (n1 + n2)
    se = np.sqrt(p * (1 - p) * (1 / n1 + 1 / n2)); z = (p1 - p2) / se
    return p1, p2, z, 2 * stats.norm.sf(abs(z))
yc, yd = new_cases_all.sum(axis=1).groupby(new_cases_all.index.year).sum(), new_deaths_all.sum(axis=1).groupby(new_deaths_all.index.year).sum()
p20, p22, z, pz = two_prop_z(yd[2020], yc[2020], yd[2022], yc[2022])
print(f"H0: CFR(2020) = CFR(2022) -> {p20:.2%} vs {p22:.2%}, z = {z:.1f}, p = {pz:.2e}", "=> reject H0" if pz < .05 else "=> fail to reject")
print("Caveat: cases are not independent trials and testing intensity changed; the test shows direction, not a causal estimate.\n")

# C. Kruskal-Wallis across US states (county rates)
cty_s = cty[~cty.Province_State.isin(TERRITORIES)]
groups = [g.deaths_per_100k.values for _, g in cty_s.groupby("Province_State") if len(g) >= 3]
H, pk = stats.kruskal(*groups)
print(f"H0: county death rates equal across states -> H = {H:.0f}, p = {pk:.2e}, groups = {len(groups)}", "=> reject H0" if pk < .05 else "=> fail to reject")
print(f"State rates range {states.deaths_per_100k.min():.0f}-{states.deaths_per_100k.max():.0f} per 100k\n")

# D. Trend significance around peaks (Mann-Kendall via Kendall tau + OLS slope)
def mk(s):
    s = s.dropna(); t = np.arange(len(s))
    tau, p = stats.kendalltau(t, s.values); lr = stats.linregress(t, s.values)
    return tau, p, lr.slope
def around_peak(name, s, win=45):
    pk = s.idxmax(); rows = []
    for phase, seg in [("rise", s[pk - pd.Timedelta(days=win):pk]), ("fall", s[pk:pk + pd.Timedelta(days=win)])]:
        tau, p, slope = mk(seg); rows.append({"series": name, "peak": pk.date(), "phase": phase, "tau": tau, "p_value": p, "slope_per_day": slope})
    return rows
trend = pd.DataFrame(sum([around_peak("World cases (Omicron)", g_cases7["2021-11-01":]),
                          around_peak("World deaths (winter 2020-21)", g_deaths7[:"2021-06-30"]),
                          around_peak("India cases (Delta)", cases7[IN]["2021-03-01":"2021-08-31"]),
                          around_peak("India cases (Omicron)", cases7[IN]["2021-12-15":"2022-03-31"])], []))
display(trend.style.format({"tau": "{:.2f}", "p_value": "{:.1e}", "slope_per_day": "{:,.0f}"}))

## 6. Observations
The cell below recomputes every headline figure from the data and checks it against the values in the project README, so the narrative is always traceable to the code.

In [ ]:
yc = new_cases_all.sum(axis=1)
obs = {
    "world_cases": int(conf_all.iloc[-1].sum()), "world_deaths": int(deaths_all.iloc[-1].sum()),
    "world_cfr_%": deaths_all.iloc[-1].sum() / conf_all.iloc[-1].sum() * 100,
    "case_peak_date": g_cases7.idxmax().date(), "case_peak_7d": g_cases7.max(),
    "death_peak_date": g_deaths7.idxmax().date(), "death_peak_7d": g_deaths7.max(),
    "india_delta_peak": india_waves.loc["Delta (2021)", "peak_cases_date"], "india_delta_7d": india_waves.loc["Delta (2021)", "peak_cases_7d"],
    "india_omicron_peak": india_waves.loc["Omicron (2022)", "peak_cases_date"], "india_omicron_7d": india_waves.loc["Omicron (2022)", "peak_cases_7d"],
}
readme = {"world_cases": 676_570_149, "case_peak_date": pd.Timestamp("2022-01-24").date(), "death_peak_date": pd.Timestamp("2021-01-26").date(),
          "india_delta_peak": pd.Timestamp("2021-05-08").date(), "india_omicron_peak": pd.Timestamp("2022-01-25").date()}
for k, v in obs.items():
    flag = "" if k not in readme else ("  [matches README]" if readme[k] == v else f"  [README says {readme[k]}]")
    print(f"{k:22s} {v:,.2f}{flag}" if isinstance(v, float) else f"{k:22s} {v:,}{flag}" if isinstance(v, int) else f"{k:22s} {v}{flag}")

print("\nTop 5 cases :", (last_c.nlargest(5) / 1e6).round(1).to_dict())
print("Top 5 deaths:", (last_d.nlargest(5) / 1e3).round(0).to_dict())
print("CFR extremes (>1M cases):", {"highest": (cfr_last.nlargest(3) * 100).round(2).to_dict(), "lowest": (cfr_last.nsmallest(3) * 100).round(2).to_dict()})
print("Top 4 US states per 100k:", states.deaths_per_100k.head(4).round(0).to_dict(), "| lowest:", states.deaths_per_100k.tail(1).round(0).to_dict())

### Key findings (answers to the five questions)
1. **Global totals and waves.** The cell above gives the totals (about 676.6 M cases, 6.88 M deaths, CFR about 1%). Cases peaked with **Omicron in January 2022**, deaths peaked a year earlier in **January 2021**. The deadliest wave was not the largest one.
2. **By year.** 2022 had by far the most cases but far fewer deaths per case than 2020 (see the yearly table and z-test). Vaccination, prior infection, a milder variant and wider testing all contribute; this analysis cannot separate them.
3. **Countries.** The US, India, France, Germany and Brazil lead on cases; the US, Brazil, India, Russia and Mexico lead on deaths. CFR varies by more than an order of magnitude (Peru, Mexico, Ecuador high; Singapore, South Korea, New Zealand low), reflecting testing, age structure and health-system capacity as well as true lethality.
4. **India.** Delta (peak early May 2021) was lower in raw cases than Omicron's global peak but far deadlier per case; the Omicron wave (peak late Jan 2022) had comparable case counts with a fraction of the deaths.
5. **Case-to-death lag.** The cross-correlation analysis quantifies the lag; correlation is weak globally because the case-to-death relationship changed across variants and vaccination, and because countries' waves were out of phase.
6. **US states.** Death rates differ significantly across states (Kruskal-Wallis); Arizona, Oklahoma, Mississippi and West Virginia sit at the top, Hawaii at the bottom among the 50 states.
7. **Data caveats.** Recovered is unusable after Aug 2021 so no active-case estimate; deaths > cases in a few countries; reporting practices and testing differ; cumulative corrections were clipped. Confirmed counts understate true infections, especially early in the pandemic and in 2022.

## 7. Recommendations
1. **Measure severity, not only cases.** Dashboards should lead with deaths, hospitalisations and CFR; case counts were a poor guide to burden in 2022.
2. **Build early-warning capacity.** Deaths trail cases by roughly 1-3 weeks, so surge plans should trigger on case growth.
3. **Learn from low-CFR countries** (South Korea, Singapore, New Zealand): early testing, contact tracing and protecting the elderly.
4. **Target high-burden US states** for public-health investment (vaccination outreach, chronic-disease management).
5. **Data governance.** Keep consistent definitions (do not abandon series like "recovered") and publish population denominators with the data.

In [ ]:
# Write a compact summary into outputs/ for the README
with open(f"{OUT}/key_findings.md", "w") as f:
    f.write("# Key findings (generated by covid_analysis.ipynb)\n\n")
    for k, v in obs.items():
        f.write(f"- **{k}**: {v:,.2f}\n" if isinstance(v, float) else f"- **{k}**: {v:,}\n" if isinstance(v, int) else f"- **{k}**: {v}\n")
    f.write(f"\n- Lagged correlation (7d avg): best lag {best_sm} d, r = {lag_sm.r.max():.3f}\n")
    f.write(f"- CFR 2020 vs 2022: {p20:.2%} vs {p22:.2%} (z = {z:.1f}, p = {pz:.1e})\n")
    f.write(f"- Kruskal-Wallis across US states: H = {H:.0f}, p = {pk:.1e}\n")
    f.write("\n## Charts\n\n")
    for fn in sorted(os.listdir(OUT)):
        if fn.endswith(".png"): f.write(f"![{fn}]({fn})\n\n")
print("Saved:", sorted(os.listdir(OUT)))